In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

# =====================================================================
# TUNED XGBOOST — same 9 early-windows as your RF/DT sweeps
# =====================================================================
# Fixes two issues from the original script:
#   1. Path bug — original used a flat path (C:\mit_supercloud_raw\model_a_...)
#      but your files actually live in per-window subfolders. Now matches
#      the same {WINDOW_SEC}s\ pattern your RF and DT scripts already use.
#   2. No tuning — original used one fixed hyperparameter set for every
#      window. This version runs RandomizedSearchCV per window, same as
#      RF and DT, so the three-way comparison in the paper is actually fair.
# Also restructured as a loop instead of 9 copy-pasted blocks, removing
# the risk of copy-paste bugs (like the stale ":.1f0" typo you caught earlier).

WINDOW_SECS = [10, 30, 60, 120, 180, 240, 300, 360, 600]

FEATURES = [
    "early_power_draw_W", "early_utilization_gpu_pct", "early_utilization_memory_pct",
    "early_memory_used_MiB", "early_memory_free_MiB",
    "early_temperature_gpu", "early_temperature_memory",
]
TARGET = "full_temperature_gpu_peak"

# Reasonable, standard XGBoost tuning search space — mirrors the scope of
# the RF/DT grids (tree depth/complexity + boosting-specific parameters).
xgb_param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 4, 5, 6, 8, 10],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.1, 0.3, 0.5],
}

all_results = []  # collect one row per window for the final summary table

for WINDOW_SEC in WINDOW_SECS:

    DATA_PATH = rf"C:\mit_supercloud_raw\{WINDOW_SEC}s\model_a_early_{WINDOW_SEC}sec_vs_full_v4.csv"
    df = pd.read_csv(DATA_PATH)
    print(f"\n{'#' * 60}")
    print(f"XGBOOST — {WINDOW_SEC}-second early window")
    print(f"{'#' * 60}")
    print(f"Loaded {len(df)} rows, {df['id_job'].nunique()} unique jobs")

    X = df[FEATURES]
    y = df[TARGET]

    # Same split settings as RF/DT scripts, so results stay directly comparable.
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
    print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")

    # Same search budget as RF (100 iterations, 5-fold CV) — this is the part
    # that was missing before. Expect this loop to take noticeably longer
    # than the old fixed-hyperparameter version.
    search = RandomizedSearchCV(
        XGBRegressor(random_state=42, n_jobs=-1),
        param_distributions=xgb_param_dist,
        n_iter=100,
        cv=5,
        scoring="r2",
        n_jobs=-1,
        random_state=42,
        verbose=0,
    )
    search.fit(X_train, y_train)
    model = search.best_estimator_

    print(f"Best XGB params: {search.best_params_}")
    print(f"Best XGB CV R²:  {search.best_score_:.3f}")

    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("\n" + "=" * 50)
    print(f"XGBOOST — RESULTS ({WINDOW_SEC}-second early window)")
    print("=" * 50)
    print(f"RMSE : {rmse:.2f} °C")
    print(f"MAE  : {mae:.2f} °C")
    print(f"R²   : {r2:.3f}")

    importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
    print("\nFeature importance:")
    print(importances.to_string())

    plt.figure(figsize=(8, 5))
    importances.plot(kind="barh")
    plt.xlabel("Importance")
    plt.title(f"XGBoost ({WINDOW_SEC}-sec Early Window) — Feature Importance")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(rf"C:\mit_supercloud_raw\{WINDOW_SEC}s\model_a_{WINDOW_SEC}s_feature_importance.png", dpi=150)
    plt.close()
    print(f"\n💾 Saved chart → model_a_{WINDOW_SEC}s_feature_importance.png")

    model.save_model(rf"C:\mit_supercloud_raw\{WINDOW_SEC}s\model_a_{WINDOW_SEC}s_xgboost.json")
    print(f"💾 Saved model → model_a_{WINDOW_SEC}s_xgboost.json")

    results_df = X_test.copy()
    results_df["actual_peak_temp"] = y_test.values
    results_df["predicted_peak_temp"] = y_pred
    results_df["error"] = results_df["predicted_peak_temp"] - results_df["actual_peak_temp"]
    results_df.to_csv(rf"C:\mit_supercloud_raw\{WINDOW_SEC}s\model_a_{WINDOW_SEC}s_test_predictions.csv", index=False)
    print(f"💾 Saved predictions → model_a_{WINDOW_SEC}s_test_predictions.csv")

    same_count = (df["early_temperature_gpu"] == df["full_temperature_gpu_peak"]).sum()
    print(f"\nFlat/idle jobs check: {same_count}/{len(df)} ({100*same_count/len(df):.1f}%)")

    all_results.append({
        "window_sec": WINDOW_SEC,
        "best_params": search.best_params_,
        "cv_r2": search.best_score_,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
    })

# =====================================================================
# SUMMARY ACROSS ALL WINDOWS — this is your Table I
# =====================================================================
summary_df = pd.DataFrame(all_results)
summary_df.to_csv(r"C:\mit_supercloud_raw\model_a_all_windows_summary.csv", index=False)
print("\n\n" + "=" * 60)
print("SUMMARY — XGBoost across all early windows")
print("=" * 60)
print(summary_df[["window_sec", "rmse", "mae", "r2"]].to_string(index=False))
print(f"\n💾 Saved summary → model_a_all_windows_summary.csv")


############################################################
XGBOOST — 10-second early window
############################################################
Loaded 2739 rows, 2508 unique jobs
Train: 2328 rows | Test: 411 rows
Best XGB params: {'subsample': 0.6, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.01, 'gamma': 0.1, 'colsample_bytree': 0.6}
Best XGB CV R²:  0.636

XGBOOST — RESULTS (10-second early window)
RMSE : 10.51 °C
MAE  : 7.12 °C
R²   : 0.599

Feature importance:
early_temperature_gpu           0.285793
early_temperature_memory        0.271389
early_power_draw_W              0.120304
early_memory_free_MiB           0.093632
early_utilization_memory_pct    0.092496
early_memory_used_MiB           0.068228
early_utilization_gpu_pct       0.068157

💾 Saved chart → model_a_10s_feature_importance.png
💾 Saved model → model_a_10s_xgboost.json
💾 Saved predictions → model_a_10s_test_predictions.csv

Flat/idle jobs check: 524/2739 (19.1%)

#########

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

# =====================================================================
# TUNED RANDOM FOREST — same 9 early-windows as your XGBoost sweep
# =====================================================================
# NOTE: your XGBoost script copy-pasted the same block 9 times (once per
# WINDOW_SEC). That works, but it's easy to introduce bugs that way — your
# 20s block actually has one (":.1f0" instead of ":.1f" in the last print,
# which will throw a ValueError if you run it). Since Random Forest tuning
# is also a lot slower than plain XGBoost (RandomizedSearchCV fits many
# forests per window), I turned this into a loop over WINDOW_SECS instead
# of 9 copy-pasted blocks — same outputs, same file naming pattern, just
# less code to accidentally break.

WINDOW_SECS = [10, 30, 60, 120, 180, 240, 300, 360, 600]

FEATURES = [
    "early_power_draw_W", "early_utilization_gpu_pct", "early_utilization_memory_pct",
    "early_memory_used_MiB", "early_memory_free_MiB",
    "early_temperature_gpu", "early_temperature_memory",
]
TARGET = "full_temperature_gpu_peak"

# Same search space as your tuned RF script before — this is what gets
# searched fresh for EVERY window, since the best tree settings for a 10s
# window won't necessarily be the best for a 600s window.
rf_param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [5, 8, 10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None],
}

all_results = []  # collect one row per window so we can build a summary table/plot at the end

for WINDOW_SEC in WINDOW_SECS:

    DATA_PATH = rf"C:\mit_supercloud_raw\{WINDOW_SEC}s\model_a_early_{WINDOW_SEC}sec_vs_full_v4.csv"
    df = pd.read_csv(DATA_PATH)
    print(f"\n{'#' * 60}")
    print(f"RANDOM FOREST — {WINDOW_SEC}-second early window")
    print(f"{'#' * 60}")
    print(f"Loaded {len(df)} rows, {df['id_job'].nunique()} unique jobs")

    X = df[FEATURES]
    y = df[TARGET]

    # Same split settings (test_size=0.15, random_state=42) as your XGBoost
    # script, so results across models/windows stay directly comparable.
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
    print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")

    # Search for this window's best RF settings. n_iter=100, cv=5 means
    # 500 forests get trained per window — this is the slow part, so expect
    # this loop to take noticeably longer than the plain-XGBoost version did.
    search = RandomizedSearchCV(
        RandomForestRegressor(random_state=42, n_jobs=-1),
        param_distributions=rf_param_grid,
        n_iter=100,
        cv=5,
        scoring="r2",
        n_jobs=-1,
        random_state=42,
        verbose=0,   # quiet per-fold logging; we already print a clear header per window
    )
    search.fit(X_train, y_train)
    model = search.best_estimator_

    print(f"Best RF params: {search.best_params_}")
    print(f"Best RF CV R²:  {search.best_score_:.3f}")

    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("\n" + "=" * 50)
    print(f"RANDOM FOREST — RESULTS ({WINDOW_SEC}-second early window)")
    print("=" * 50)
    print(f"RMSE : {rmse:.2f} °C")
    print(f"MAE  : {mae:.2f} °C")
    print(f"R²   : {r2:.3f}")

    # Random Forest gives us feature_importances_ just like XGBoost did, so
    # this part is basically identical to your original script.
    importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
    print("\nFeature importance:")
    print(importances.to_string())

    plt.figure(figsize=(8, 5))
    importances.plot(kind="barh")
    plt.xlabel("Importance")
    plt.title(f"Random Forest ({WINDOW_SEC}-sec Early Window) — Feature Importance")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(rf"C:\mit_supercloud_raw\model_rf_{WINDOW_SEC}s_feature_importance.png", dpi=150)
    plt.close()
    print(f"\n💾 Saved chart → model_rf_{WINDOW_SEC}s_feature_importance.png")

    # RandomForestRegressor doesn't have XGBoost's built-in .save_model(), so
    # we use joblib instead — the standard way to persist sklearn models.
    joblib.dump(model, rf"C:\mit_supercloud_raw\model_rf_{WINDOW_SEC}s_random_forest.joblib")
    print(f"💾 Saved model → model_rf_{WINDOW_SEC}s_random_forest.joblib")

    results_df = X_test.copy()
    results_df["actual_peak_temp"] = y_test.values
    results_df["predicted_peak_temp"] = y_pred
    results_df["error"] = results_df["predicted_peak_temp"] - results_df["actual_peak_temp"]
    results_df.to_csv(rf"C:\mit_supercloud_raw\model_rf_{WINDOW_SEC}s_test_predictions.csv", index=False)
    print(f"💾 Saved predictions → model_rf_{WINDOW_SEC}s_test_predictions.csv")

    same_count = (df["early_temperature_gpu"] == df["full_temperature_gpu_peak"]).sum()
    print(f"\nFlat/idle jobs check: {same_count}/{len(df)} ({100*same_count/len(df):.1f}%)")

    all_results.append({
        "window_sec": WINDOW_SEC,
        "best_params": search.best_params_,
        "cv_r2": search.best_score_,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
    })

# =====================================================================
# SUMMARY ACROSS ALL WINDOWS — this is your accuracy-vs-lead-time table
# =====================================================================
summary_df = pd.DataFrame(all_results)
summary_df.to_csv(r"C:\mit_supercloud_raw\model_rf_all_windows_summary.csv", index=False)
print("\n\n" + "=" * 60)
print("SUMMARY — Random Forest across all early windows")
print("=" * 60)
print(summary_df[["window_sec", "rmse", "mae", "r2"]].to_string(index=False))
print(f"\n💾 Saved summary → model_rf_all_windows_summary.csv")

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

# =====================================================================
# TUNED LINEAR REGRESSION (Ridge vs Lasso) — same 9 early-windows
# =====================================================================
# Same loop structure as the RF/DT versions. For each window we tune BOTH
# Ridge and Lasso over a wide range of alpha (penalty strength), then keep
# whichever one scored better on cross-validation for that specific window
# — the winner isn't assumed to be the same across every window.

WINDOW_SECS = [10, 30, 60, 120, 180, 240, 300, 360, 600]

FEATURES = [
    "early_power_draw_W", "early_utilization_gpu_pct", "early_utilization_memory_pct",
    "early_memory_used_MiB", "early_memory_free_MiB",
    "early_temperature_gpu", "early_temperature_memory",
]
TARGET = "full_temperature_gpu_peak"

# Same broad log-spaced alpha range as before — covers a near-zero penalty
# all the way up to a very heavy one, since we don't know in advance how
# much regularization each window's data actually needs.
alphas = np.logspace(-6, 6, 61)

all_results = []

for WINDOW_SEC in WINDOW_SECS:

    DATA_PATH = rf"C:\mit_supercloud_raw\{WINDOW_SEC}s\model_a_early_{WINDOW_SEC}sec_vs_full_v4.csv"
    df = pd.read_csv(DATA_PATH)
    print(f"\n{'#' * 60}")
    print(f"LINEAR REGRESSION (Ridge/Lasso) — {WINDOW_SEC}-second early window")
    print(f"{'#' * 60}")
    print(f"Loaded {len(df)} rows, {df['id_job'].nunique()} unique jobs")

    X = df[FEATURES]
    y = df[TARGET]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
    print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")

    # ---- Ridge ----
    # Pipeline bundles "scale features, then fit Ridge" so scaling is redone
    # correctly inside each CV fold instead of leaking info across folds.
    ridge_pipe = Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(random_state=42))])
    ridge_search = GridSearchCV(ridge_pipe, {"ridge__alpha": alphas}, cv=5, scoring="r2", n_jobs=-1, verbose=0)
    ridge_search.fit(X_train, y_train)

    # ---- Lasso ----
    # max_iter bumped up since Lasso's solver converges slower than Ridge's.
    lasso_pipe = Pipeline([("scaler", StandardScaler()), ("lasso", Lasso(random_state=42, max_iter=10000))])
    lasso_search = GridSearchCV(lasso_pipe, {"lasso__alpha": alphas}, cv=5, scoring="r2", n_jobs=-1, verbose=0)
    lasso_search.fit(X_train, y_train)

    # Keep whichever scored better on cross-validation for THIS window.
    if ridge_search.best_score_ >= lasso_search.best_score_:
        best_search, best_name = ridge_search, "Ridge"
    else:
        best_search, best_name = lasso_search, "Lasso"

    model = best_search.best_estimator_  # full Pipeline: scaler + linear model

    print(f"Ridge best alpha: {ridge_search.best_params_['ridge__alpha']:.4g} | CV R²: {ridge_search.best_score_:.3f}")
    print(f"Lasso best alpha: {lasso_search.best_params_['lasso__alpha']:.4g} | CV R²: {lasso_search.best_score_:.3f}")
    print(f"Selected model for this window: {best_name}")

    y_pred = model.predict(X_test)  # Pipeline scales X_test automatically before predicting

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("\n" + "=" * 50)
    print(f"LINEAR REGRESSION ({best_name}) — RESULTS ({WINDOW_SEC}-second early window)")
    print("=" * 50)
    print(f"RMSE : {rmse:.2f} °C")
    print(f"MAE  : {mae:.2f} °C")
    print(f"R²   : {r2:.3f}")

    # Linear models don't have feature_importances_ — instead we look at the
    # coefficients. Because features were scaled first, these coefficients
    # ARE directly comparable to each other (unlike raw, unscaled coefficients,
    # where a feature in the thousands would look artificially "important").
    est = model.named_steps[best_name.lower()]
    coefs = pd.Series(est.coef_.ravel(), index=FEATURES).sort_values(key=np.abs, ascending=False)
    print("\nCoefficients (scaled features, sorted by magnitude):")
    print(coefs.to_string())

    plt.figure(figsize=(8, 5))
    colors = ["#C44E52" if c < 0 else "#55A868" for c in coefs.values]
    coefs.plot(kind="barh", color=colors)
    plt.xlabel("Coefficient (scaled features)")
    plt.axvline(0, color="black", linewidth=1)
    plt.title(f"{best_name} ({WINDOW_SEC}-sec Early Window) — Coefficients")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(rf"C:\mit_supercloud_raw\model_lin_{WINDOW_SEC}s_coefficients.png", dpi=150)
    plt.close()
    print(f"\n💾 Saved chart → model_lin_{WINDOW_SEC}s_coefficients.png")

    joblib.dump(model, rf"C:\mit_supercloud_raw\model_lin_{WINDOW_SEC}s_{best_name.lower()}.joblib")
    print(f"💾 Saved model → model_lin_{WINDOW_SEC}s_{best_name.lower()}.joblib")

    results_df = X_test.copy()
    results_df["actual_peak_temp"] = y_test.values
    results_df["predicted_peak_temp"] = y_pred
    results_df["error"] = results_df["predicted_peak_temp"] - results_df["actual_peak_temp"]
    results_df.to_csv(rf"C:\mit_supercloud_raw\model_lin_{WINDOW_SEC}s_test_predictions.csv", index=False)
    print(f"💾 Saved predictions → model_lin_{WINDOW_SEC}s_test_predictions.csv")

    same_count = (df["early_temperature_gpu"] == df["full_temperature_gpu_peak"]).sum()
    print(f"\nFlat/idle jobs check: {same_count}/{len(df)} ({100*same_count/len(df):.1f}%)")

    all_results.append({
        "window_sec": WINDOW_SEC,
        "selected_model": best_name,
        "ridge_alpha": ridge_search.best_params_["ridge__alpha"],
        "lasso_alpha": lasso_search.best_params_["lasso__alpha"],
        "cv_r2": best_search.best_score_,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
    })

# =====================================================================
# SUMMARY ACROSS ALL WINDOWS
# =====================================================================
summary_df = pd.DataFrame(all_results)
summary_df.to_csv(r"C:\mit_supercloud_raw\model_lin_all_windows_summary.csv", index=False)
print("\n\n" + "=" * 60)
print("SUMMARY — Linear Regression (Ridge/Lasso) across all early windows")
print("=" * 60)
print(summary_df[["window_sec", "selected_model", "rmse", "mae", "r2"]].to_string(index=False))
print(f"\n💾 Saved summary → model_lin_all_windows_summary.csv")